# Text Extraction With Regex, Gazetteer, Ollama Labels, And GLiNER

This notebook exercises text extraction techniques. It uses a fake Ollama transport and fake GLiNER model so the workflow is reproducible offline. Replace those with live `OllamaProvider` and installed `gliner` when running real experiments.

In [ ]:
from pathlib import Path
import sys, json, tempfile

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next((p for p in Path.cwd().parents if (p / 'src').exists()), Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print(f'Repo root: {ROOT}')

In [ ]:
import json
from kg_construction.infrastructure.loaders.text_loader import TextLoader
from kg_construction.infrastructure.extractors.regex import RegexEntityExtractor
from kg_construction.infrastructure.extractors.gazetteer import GazetteerEntityExtractor

source = TextLoader().load(ROOT / 'examples/data_to_kg/sample.txt')[0]
regex_batch = RegexEntityExtractor().extract(source)
gazetteer_batch = GazetteerEntityExtractor({
    'Organization': ['Platform Team'],
    'Incident': ['INCIDENT-1042'],
    'System': ['Recommendation API'],
}).extract(source)
print('Regex:', [(e.entity_type, e.name, e.evidence[0].location) for e in regex_batch.entities])
print('Gazetteer:', [(e.entity_type, e.name) for e in gazetteer_batch.entities])

In [ ]:
from kg_construction.application.hybrid_gliner import HybridGlinerPipelineConfig, extract_entities_with_ollama_labels_and_gliner
from kg_construction.infrastructure.llm.ollama import OllamaConfig, OllamaProvider

class FakeGlinerModel:
    def predict_entities(self, text, labels, *, threshold=0.5, flat_ner=True):
        spans = {
            'Incident': 'INCIDENT-1042',
            'Organization': 'Platform Team',
            'Date': '2026-07-09',
            'System': 'Recommendation API',
        }
        entities = []
        for label, value in spans.items():
            if label not in labels:
                continue
            start = text.find(value)
            if start >= 0:
                entities.append({'text': value, 'label': label, 'start': start, 'end': start + len(value), 'score': 0.91})
        return entities

def fake_transport(_body):
    return {'message': {'content': json.dumps({'labels': [
        {'name': 'Incident', 'confidence': 0.95, 'description': 'Operational incident identifier'},
        {'name': 'Organization', 'confidence': 0.9},
        {'name': 'Date', 'confidence': 0.85},
        {'name': 'System', 'confidence': 0.85},
        {'name': 'Thing', 'confidence': 0.2},
    ]})}}

config = HybridGlinerPipelineConfig.from_yaml(ROOT / 'configs/ollama_gliner_label_proposal.example.yaml').model_copy(update={'compare_ollama_graph': False})
provider = OllamaProvider(OllamaConfig(max_retries=0), transport=fake_transport)
result = extract_entities_with_ollama_labels_and_gliner(
    source.content or '',
    config=config,
    provider=provider,
    gliner_model=FakeGlinerModel(),
)
print('Reviewed labels:', result.reviewed_labels)
print('GLiNER entities:', [(e.entity_type, e.name, e.evidence[0].location) for e in result.gliner_batch.entities])
print('Comparison:', [row.model_dump() for row in result.comparisons])